In [1]:
#overview
"""
This code will have a lot of technical details so I will provide a brief overview of what I am trying to do here:
1. Import all the necessary libraries
2. Set the parameters and initialize the dataframe
3. Calculate the nuclear timescale (M/L) as an order-of-magnitude estimate for comparison with the final integrated lifetime
4. Use the Lane-Emden polytrope equation to define a function for pressure and temperature at a given radius R
5. Define the equation used to determine the reaction rate for the proton-proton chain
6. Integrate over the radius of the star using the pressure and temperature functions from (4) as inputs to the equation in (5)
and multiply by the energy per reaction to get a luminosity
7. Multiply luminosity by the "bin width", and calculate how much total energy was used over this timespan
8. Determine the change in hydrogen mass fraction needed to produce said energy
9. Update the data table with the change in hydrogen mass fraction
10. Repeat steps 4-9 in a recursive algorithm until the hydrogen reaches 0
11. Multiply the length of the data frame by the bin width to get an expected lifespan
12. Compare the result to the prediction made in (3) to get an error value
13. A visualization of the simulation will be available after the algorithm terminates
"""

'\nThis code will have a lot of technical details so I will provide a brief overview of what I am trying to do here:\n1. Import all the necessary libraries\n2. Set the parameters and initialize the dataframe\n3. Calculate the nuclear timescale (M/L) as an order-of-magnitude estimate for comparison with the final integrated lifetime\n4. Use the Lane-Emden polytrope equation to define a function for pressure and temperature at a given radius R\n5. Define the equation used to determine the reaction rate for the proton-proton chain\n6. Integrate over the radius of the star using the pressure and temperature functions from (4) as inputs to the equation in (5)\nand multiply by the energy per reaction to get a luminosity\n7. Multiply luminosity by the "bin width", and calculate how much total energy was used over this timespan\n8. Determine the change in hydrogen mass fraction needed to produce said energy\n9. Update the data table with the change in hydrogen mass fraction\n10. Repeat steps 4

In [2]:
#importing libraries
import pandas as pd
import matplotlib.pyplot as plt
import scipy as si
import numpy as np
import matplotlib.patches as patches
from matplotlib.widgets import Slider
from scipy.integrate import solve_ivp, simpson
from astropy import constants as const
from astropy import units as u

In [11]:
#part here to initialize the data frame
"""
parameters
-mass of the star (initial_mass)
-time since simulation started (time_elapsed)
-mass fraction of hydrogen (X)
-mass fraction of helium (Y)
-metallicity (mass fraction of other elements) (Z)
-bin width (bin_width)

assumptions
-Constant mass, luminosity, and radius
-Only red dwarves (100% convective, only P-P chain, only hydrogen fusion)

Users can set the metallicity and mass at the start
"""
bin_width = 10e6 #discrete period over which the simulation runs, in years

time = 0
X = 0.75
Y = 0.25
Z = 0.0
initial_mass = 0.5 #a value between 0.1 and 0.5 solar massess, the mass range for fully convective red dwarfs

#some physical constants from astropy
M_sun = const.M_sun.cgs.value      # g
R_sun = const.R_sun.cgs.value      # cm
L_sun = const.L_sun.cgs.value      # erg/s
G = const.G.cgs.value              # cm³/g/s²
k_B = const.k_B.cgs.value          # erg/K
m_p = const.m_p.cgs.value          # g
c = const.c.cgs.value              # cm/s

columns = ['time', 'X', 'Y', 'Z', 'mass', 'T_core', 'rho_core', 'L', 'R'] #The last 4 variables will be needed for the algorithm and are dependent
initial_data = {
    'time': [time],
    'X': [X],
    'Y': [Y],
    'Z': [Z],
    'mass': [initial_mass],
    'T_core': [np.nan],    # Will calculate in first step
    'rho_core': [np.nan],  # Will calculate in first step
    'L': [np.nan],         # Will calculate in first step
    'R': [np.nan]          # Will calculate in first step
}
main_df = pd.DataFrame(initial_data)

In [4]:
#Control output 
"""
This cell runs predictions using the sun's known lifespan, the nuclear timescale, and the mass-luminosity relationship of stars to determine
the expected lifespan of the star.

The nuclear timescale uses the fusion rate within a star to relate luminosity to mass. The star "dies" when the fuel has been exhausted,
and thus scales as M/L, or how much the star radiates vs. how much fuel it can burn.
If the luminosity of the star stays constant throughout its lifetime, then luminosity scales roughly as L ~ M^3.5.
(Important note on the above point: the luminosity of the star will drastically increase during the giant phases, so the nuclear timescale
values are typically said to give the main-sequence lifespan of a star, though for the purposes of this project, that will not matter so much,
since red dwarfs never enter the giant phase of their life.)
We can also assume the fuel the star has scales LOCALLY (<0.5 M_sun) proportionally to M. An important nuance here is that the sun, the benchmark star,
does NOT have a fully convective interior, and thus only fuses about 10% of its available hydrogen, whereas a red dwarf is fully convective
and will use all of its available fuel. We need to correct by multiplying the nuclear timescale by 10.

Putting these components together, we get that lifespan goes as M/M^3.5 ~ M^-2.5.

The sun has an expected lifespan of 10^10yr (10Gyr). We can use a simple ratio that takes the mass of the input star and returns its expected lifespan.
"""
def nuclear_timescale(mass):
    return (1 / (mass ** 2.5)) * 1e10 * 10

print(f"{nuclear_timescale(initial_mass):e}")

2.028602e+12


In [5]:
#Lane-Emden Equation Class - First component of solving for energy

"""
Model approach: Hydrostatic equilibrium (polytrope)+ energy (nuclear timescale)
To maintain hydrostatic equilibrium, the star must have a certain amount of pressure counteracting gravity at a given radius
We can calculate how much energy is released from each of the main reaction types, and how many of those reactions are occuring

defining functions to update the data frame
lane-emden equations with polytrope index n = 1.5 (appropriate for red dwarf stars)
use polytropic approach for simplifications; need definite integrals
use Runge-Kutta to solve the differential equation

IMPORTANT: Credit to Michael Zingale, whose notebook I referenced for the derivation and code at this link: https://zingale.github.io/stars/notebooks/polytrope/lane-emden.html
"""
class LaneEmdenSolver: #Solves the Lane-Emden equation for a polytrope of index n
    
    def __init__(self, n=1.5):
        self.n = n
        self.xi = None
        self.theta = None
        self.dtheta_dxi = None
        self.xi_surface = None 
    def solve(self, xi_max=10):
        """Solve the Lane-Emden equation from center to surface."""
        # Initial conditions: θ(0)=1, θ'(0)=0
        y0 = [1.0, 0.0]
        
        # Event: stop when theta crosses zero (surface)
        def surface_event(xi, y):
            return y[0]  # theta = 0
        surface_event.terminal = True
        surface_event.direction = -1
        
        # Solve using scipy
        sol = solve_ivp(
            fun=self._lane_emden_derivatives,
            t_span=[1e-6, xi_max],
            y0=y0,
            method='RK45',
            events=surface_event,
            dense_output=True,
            max_step=0.1
        )
        
        # Extract solution
        self.xi = sol.t
        self.theta = sol.y[0]
        self.dtheta_dxi = sol.y[1]
        
        # CRITICAL: Remove any negative theta values
        positive_mask = self.theta > 0
        self.xi = self.xi[positive_mask]
        self.theta = self.theta[positive_mask]
        self.dtheta_dxi = self.dtheta_dxi[positive_mask]
        
        self.xi_surface = self.xi[-1]
        
        print(f"Kept {len(self.xi)} points with positive theta")
        
        return self.xi, self.theta, self.dtheta_dxi
    
    def _lane_emden_derivatives(self, xi, y):
        """Lane-Emden equation derivatives."""
        theta, dtheta = y
        
        # Ensure theta stays positive
        theta = max(theta, 0.0)
        
        # Handle singularity at xi=0
        if xi < 1e-10:
            d2theta = -theta**self.n / 3.0
        else:
            d2theta = -2.0*dtheta/xi - theta**self.n
        
        return [dtheta, d2theta]


def get_stellar_structure(M_solar, X, polytrope_solution):
    """
    Convert Lane-Emden solution to physical stellar structure.
    
    Parameters:
    -----------
    M_solar : float
        Stellar mass in solar masses
    X : float
        Hydrogen mass fraction
    polytrope_solution : LaneEmdenSolver
        Solved Lane-Emden equation
    
    Returns:
    --------
    dict with keys: 'R', 'rho_c', 'T_c', 'r', 'rho', 'T'
    """
    M = M_solar * M_sun
    
    # Mean molecular weight for fully ionized gas
    Y = 1.0 - X
    mu = 1.0 / (2.0*X + 0.75*Y)
    
    # Mass-radius relation for red dwarfs
    R = (M_solar**0.8) * R_sun
    
    # Scaling factor
    alpha = R / polytrope_solution.xi_surface
    
    # Central density from mass normalization
    integrand = polytrope_solution.xi**2 * polytrope_solution.theta**polytrope_solution.n
    integral = simpson(integrand, polytrope_solution.xi)
    rho_c = M / (4.0 * np.pi * alpha**3 * integral)
    
    # Central temperature
    T_virial = G * M * mu * m_p / (k_B * R)
    T_c = 0.6 * T_virial  # Correction factor for n=1.5
    
    # Full profiles
    r = alpha * polytrope_solution.xi
    rho = rho_c * polytrope_solution.theta**polytrope_solution.n
    T = T_c * polytrope_solution.theta
    
    return {
        'R': R,
        'rho_c': rho_c,
        'T_c': T_c,
        'r': r,
        'rho': rho,
        'T': T
    }

# Test the solver
if __name__ == "__main__":
    # Solve Lane-Emden for n=1.5
    solver = LaneEmdenSolver(n=1.5)
    xi, theta, dtheta = solver.solve()
    
    print(f"Lane-Emden solution for n=1.5:")
    print(f"  Surface at ξ = {solver.xi_surface:.4f}")
    print(f"  Expected: ξ ≈ 3.65 for n=1.5")
    
    # Get structure for 0.3 solar mass red dwarf
    structure = get_stellar_structure(0.3, X=0.75, polytrope_solution=solver)
    
    print(f"\nStellar structure for 0.3 M☉:")
    print(f"  Radius: {structure['R']/R_sun:.3f} R☉")
    print(f"  Central density: {structure['rho_c']:.3e} g/cm³")
    print(f"  Central temperature: {structure['T_c']/1e6:.2f} million K")

Kept 44 points with positive theta
Lane-Emden solution for n=1.5:
  Surface at ξ = 3.5665
  Expected: ξ ≈ 3.65 for n=1.5

Stellar structure for 0.3 M☉:
  Radius: 0.382 R☉
  Central density: 4.240e+01 g/cm³
  Central temperature: 6.46 million K


In [6]:
#The energy generated by the proton-proton chain
"""
Equation derived from this Harvard lecture I found, linked here: https://people.ast.cam.ac.uk/~phewett/SandES2017/Lec2017_11.pdf

"""
def epsilon_pp(rho, T, X):
    """
    PP-chain energy generation rate (CGS units)
    
    From: Clayton (1968), Kippenhahn & Weigert (1990)
    """
    epsilon_0 = 2.4e4  # erg/g/s (CGS)
    T7 = T / 1e7       # Temperature in units of 10^7 K
    
    return epsilon_0 * rho * X**2 * T7**4

In [19]:
#run the recursive algorithm
"""

"""

def update_composition_simple(L_solar, M_solar, X, Y, dt_years):
    """
    Simpler version: use mass-energy conversion directly.
    
    Parameters:
    -----------
    L_solar : float
        Luminosity in solar luminosities
    M_solar : float
        Stellar mass in solar masses
    X, Y : float
        Current hydrogen and helium fractions
    dt_years : float
        Timestep in years
    
    Returns:
    --------
    X_new, Y_new : float
        Updated mass fractions
    """
    # Energy radiated over timestep
    L_erg_s = L_solar * L_sun
    dt_sec = dt_years * 3.156e7
    E_radiated = L_erg_s * dt_sec  # erg
    
    # Mass-energy relation: E = Δm * c²
    # For H→He: efficiency η = 0.007 (0.7% of mass converts to energy)
    eta = 0.007
    
    # Total hydrogen mass consumed
    M_H_consumed = E_radiated / (eta * c**2)  # grams
    
    # Change in mass fraction
    M_star = M_solar * M_sun
    dX = -M_H_consumed / M_star
    
    # Update
    X_new = X + dX
    Y_new = Y - dX
    
    # Don't let X go negative
    if X_new < 0:
        X_new = 0
        Y_new = 1.0
    
    return X_new, Y_new

def calculate_luminosity(structure, X):
    """
    Calculate stellar luminosity - CORRECTED VERSION
    """
    r = structure['r']
    rho = structure['rho']
    T = structure['T']
    
    # PP-chain formula from Padmanabhan "Theoretical Astrophysics Vol II"
    # ε_pp = 0.241 ρ X² f_pp(T) where f_pp depends on temperature
    
    # For T = 10^6 to 10^7 K (red dwarf range):
    # f_pp ≈ (T/T_0)^ν where T_0 = 1.5×10^7 K, ν ≈ 4
    
    T_0 = 1.5e7  # Reference temperature
    nu = 4.0
    
    # The constant in CGS: erg/g/s
    epsilon_0 = 0.241  # This is for ρ in g/cm³, T in ratio form
    
    epsilon_pp = epsilon_0 * rho * X**2 * (T/T_0)**nu
    
    # Integrate
    integrand = 4.0 * np.pi * r**2 * rho * epsilon_pp
    L_total = simpson(integrand, r)
    L_solar = L_total / L_sun
    
    return L_solar

# Evolution loop
step = 0
max_steps = 10000  # Safety limit
dt = bin_width

# RESET everything before evolution
time = 0
X = 0.75
Y = 0.25
Z = 0.0
initial_mass = 0.5
dt = 1e8  # 100 million years

step = 0
max_steps = 10000

# Calculate luminosity
L = calculate_luminosity(structure, X)
print(f"LUMINOSITY: {L:.6e} L_sun")
print(f"Expected for 0.3 M_sun: ~0.01 L_sun")

# Also check the structure values
print(f"T_core: {structure['T_c']/1e6:.2f} MK")
print(f"rho_core: {structure['rho_c']:.2e} g/cm³")
print(f"Number of radial points: {len(structure['r'])}")

while X > 0.005 and step < max_steps:
    # Get current stellar structure
    structure = get_stellar_structure(initial_mass, X, solver)
    
    # Calculate luminosity
    L = calculate_luminosity(structure, X)
    
    # Update composition
    X_new, Y_new = update_composition_simple(L, initial_mass, X, Y, dt)
    
    # Update time
    time += dt
    
    # Store in dataframe
    new_row = {
        'time': time / 1e9,  # Convert to Gyr
        'X': X_new,
        'Y': Y_new,
        'Z': Z,
        'mass': initial_mass,
        'T_core': structure['T_c'],
        'rho_core': structure['rho_c'],
        'L': L,
        'R': structure['R'] / R_sun
    }
    main_df = pd.concat([main_df, pd.DataFrame([new_row])], ignore_index=True)
    
    # Update for next iteration
    X = X_new
    Y = Y_new
    step += 1
    
    # Progress update
    if step % 100 == 0:
        print(f"Step {step}: t={time/1e9:.2f} Gyr, X={X:.4f}, L={L:.6f} L☉")

print(f"\nEvolution complete!")
print(f"Total time: {time/1e9:.1f} Gyr")
print(f"Final X: {X:.4f}")
print(f"Steps taken: {step}")

LUMINOSITY: 4.083965e-03 L_sun
Expected for 0.3 M_sun: ~0.01 L_sun
T_core: 5.87 MK
rho_core: 1.97e+02 g/cm³
Number of radial points: 44
Step 100: t=10.00 Gyr, X=0.7491, L=0.004728 L☉
Step 200: t=20.00 Gyr, X=0.7482, L=0.004729 L☉
Step 300: t=30.00 Gyr, X=0.7473, L=0.004730 L☉
Step 400: t=40.00 Gyr, X=0.7463, L=0.004732 L☉
Step 500: t=50.00 Gyr, X=0.7454, L=0.004733 L☉
Step 600: t=60.00 Gyr, X=0.7445, L=0.004734 L☉
Step 700: t=70.00 Gyr, X=0.7436, L=0.004736 L☉
Step 800: t=80.00 Gyr, X=0.7427, L=0.004737 L☉
Step 900: t=90.00 Gyr, X=0.7418, L=0.004738 L☉
Step 1000: t=100.00 Gyr, X=0.7409, L=0.004739 L☉
Step 1100: t=110.00 Gyr, X=0.7399, L=0.004740 L☉
Step 1200: t=120.00 Gyr, X=0.7390, L=0.004742 L☉
Step 1300: t=130.00 Gyr, X=0.7381, L=0.004743 L☉
Step 1400: t=140.00 Gyr, X=0.7372, L=0.004744 L☉
Step 1500: t=150.00 Gyr, X=0.7363, L=0.004745 L☉
Step 1600: t=160.00 Gyr, X=0.7354, L=0.004747 L☉
Step 1700: t=170.00 Gyr, X=0.7344, L=0.004748 L☉
Step 1800: t=180.00 Gyr, X=0.7335, L=0.004749 L☉


In [ ]:
#code to produce the real-time visuals
"""
this part will work by plotting the stellar core on a 2D cartesian grid as rings of elements
as a simplification (just for visuals), the radius of each ring is just linearly proportional to the fraction of elements, instead of calculating volume
i.e.
r1 = X+Y+Z, r2 = Y+Z, r3 = Z
how to make a real-time sim? how to make a slider to go back and forth on the timeline?
how to plot circle & change color? likely matplotlib library
"""
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.widgets import Slider
import numpy as np

# Example data - replace with your actual simulation data
# Each row: [time (in Myr), X, Y, Z]
simulation_data = np.array([
    [0, 0.70, 0.28, 0.02],
    [100, 0.65, 0.32, 0.03],
    [200, 0.60, 0.36, 0.04],
    [300, 0.55, 0.40, 0.05],
    [400, 0.50, 0.44, 0.06],
    [500, 0.45, 0.47, 0.08],
    [600, 0.40, 0.50, 0.10],
])

# Enable interactive mode
plt.ion()

def plot_core(time_index, ax):
    """Plot the stellar core composition at a given time index"""
    ax.clear()
    
    # Get data for this timestep
    time, X, Y, Z = simulation_data[time_index]
    
    # Calculate radii (based on your spec)
    r1 = X
    r2 = X + Y
    r3 = 1.0  # X + Y + Z should equal 1
    
    # Create circles (plot from outside-in for proper layering)
    # Metals (outermost)
    circle3 = patches.Circle((0, 0), r3, color='blue', label='Metals (Z)')
    ax.add_patch(circle3)
    
    # Helium (middle)
    circle2 = patches.Circle((0, 0), r2, color='orange', label='Helium (Y)')
    ax.add_patch(circle2)
    
    # Hydrogen (innermost)
    circle1 = patches.Circle((0, 0), r1, color='red', label='Hydrogen (X)')
    ax.add_patch(circle1)
    
    # Set axis properties
    ax.set_xlim(-1.2, 1.2)
    ax.set_ylim(-1.2, 1.2)
    ax.set_aspect('equal')
    ax.set_title(f'Stellar Core Composition at t={time:.1f} Myr', fontsize=14)
    ax.legend(loc='upper right')
    
    # Hide axes
    ax.axis('off')
    
    # Add text showing fractions
    ax.text(0, -1.4, f'X={X:.3f}, Y={Y:.3f}, Z={Z:.3f}', 
            ha='center', fontsize=10)

# Setup the figure and axis
fig = plt.figure(figsize=(8, 9))
ax = fig.add_axes([0.1, 0.25, 0.8, 0.65])  # [left, bottom, width, height]

# Initial plot
plot_core(0, ax)

# Create slider for timeline scrubbing
ax_slider = fig.add_axes([0.2, 0.1, 0.6, 0.03])
time_slider = Slider(
    ax=ax_slider,
    label='Time Step',
    valmin=0,
    valmax=len(simulation_data) - 1,
    valinit=0,
    valstep=1
)

# Update function for slider
def update(val):
    time_index = int(time_slider.val)
    plot_core(time_index, ax)
    fig.canvas.draw_idle()

time_slider.on_changed(update)

plt.show()